#  <center> Problem Set 4 (JAK2) <center>
<center> Spring 2025 <center>
<center> 7.C01/7.C51, 20.C01/20.C51 <center>
<center> Due: Monday, April 28, 2025 at 3:00 PM ET. <center>

<b>Name:</b>

<b>Kerberos ID:</b>

## Instructions

Put your code in the code blocks flagged with `########## Code ##########`.

Numerical answers yielded from running the code should be included in an Answer Block (see next cell).

We have provided print statements where numerical answers are expected.

Your answer should be contained in a variable which you defined either in the Answer Block or the Code Block.

When a qualitative answer is expected, place those comments as Markdown/Text cells; when asked for within Code blocks, you can write answer as code comments by placing a # before your answer.

Your Answer Block should look like the following:

In [ ]:
########## Answer ##########

ans = 2
print("My answer is: {}.".format(ans))

# My regressor over-fitted the training data, I need to add regularization

########## Answer ##########

## Imports


In [ ]:
!pip install rdkit
!pip install molvs

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import DBSCAN

from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator
from molvs import standardize_smiles

## Grading guideline

- Didn't answer the question 0%
- Showed some attempts, but clearly didn't try enough: 25%
- Showed solid attempts (showed code) but does not answer the question directly: 50%
- Showed solid attempts and get the question wrong: 60-80%
- Showed solid attempts with some small mistakes: 80-90%
- Showed code and answered the questions correctly: 100%

## Download required data

In [ ]:
!wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps4-bio/data/jak2.csv

## Part 1: Dimensionality Reduction for Molecular Representations

### 1.1 (10 points, Grad only) Choosing radius/bits for Morgan fingerprints

Provide a one-sentence description of what the radius represents and another of what the number of bits represents. How does adjusting the radius parameter affect the granularity of the motifs captured by the fingerprints, and how does this relate to the choice of the number of bits?

**Answer:**

### 1.2 (15 points) Principal Component Analysis on Molecular Fingerprints

In [ ]:
########## Run ##########

# convert SMILES strings to Morgan fingerprints with rdkit
jak2 = pd.read_csv("jak2.csv")
radius = 3
num_bits = 2048

class ECFP:
    def __init__(self, smiles):
        self.mols = [Chem.MolFromSmiles(i) for i in smiles]
        self.smiles = smiles

    def compute_ECFP(self):
        bit_headers = ["bit" + str(i) for i in range(num_bits)]
        arr = np.empty((0, num_bits), int).astype(int)
        mol_all = []
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius,
                                                           fpSize=num_bits)
        for i in self.mols:
            mol_all.append(i)
            fp = mfpgen.GetFingerprint(i)
            arr = np.vstack((arr, fp))

        df_fp = pd.DataFrame(np.asarray(arr).astype(int),columns=bit_headers)
        df_fp.insert(loc=0, column="smiles", value=self.smiles)
        df_fp.insert(loc=1, column="mol", value=mol_all)

        return df_fp

smiles_standarized = [standardize_smiles(i) for i in jak2["SMILES"].values]
jak2_fp_descriptor = ECFP(smiles_standarized)
jak2_fp = jak2_fp_descriptor.compute_ECFP()

# remove first column as we will reference smiles column from "jak2" dataframe
jak2_fp = jak2_fp.drop(columns=["smiles", "mol"])  # second/third not needed

########## Run ##########

This resulting dataframe, `jak2_fp`, contains the 2048 bits (columns) making up the fingerprints for the 1,911 molecules (rows). Now, perform PCA to reduce data into vectors of 100 dimensions.

In [ ]:
########## Code ##########


########## Code ##########

Visualize the first two components of your data in a 2D scatter plot and color each molecule by its pIC50.

In [ ]:
########## Code ##########

# skeleton code for plotting
fig, ax = plt.subplots(figsize=(5,5))

sc = ax.scatter(, , s=3, c=jak2["pIC50"], cmap="viridis")
cbar = plt.colorbar(sc)
cbar.set_label("pIC50", rotation=270, labelpad=15)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.show()

########## Code ##########

What is the percent explained variance of the first 50 principal components?

In [ ]:
########## Answer ##########


########## Answer ###########

What patterns do you observe (if any)?

**Answer:**

### 1.3 (15 points) t-SNE analysis on Molecular Fingerprints

Perform t-SNE on the obtained principal components, with perplexity value of 2, 30, and 500. Plot the results and label your plots.

In [ ]:
########## Code ##########

fig, ax = plt.subplots(figsize=(18,6), ncols=4, gridspec_kw={"width_ratios": [1,1,1,0.05]})

perplexities =
for i, perplexity in enumerate(perplexities):

    ax[i].scatter( , , s=3, c=jak2["pIC50"])
    ax[i].set_xlabel("t-SNE1")
    ax[i].set_ylabel("t-SNE2")
    ax[i].set_title(f"Perplexity = {perplexity}")

cbar = plt.colorbar(sc, cax=ax[-1])
cbar.set_label("pIC50", rotation=270, labelpad=15)

plt.tight_layout()
plt.show()

########## Code ##########

What differences do you see between the three t-SNE plots? What patterns do you observe in the `perplexity=30` plot?

**Answer:**

### 1.4 (35 points) Are the low dimensional embeddings meaningful?

Discretize pIC50 data by classifying any molecule with a `pIC50 >= 9.5` as effective (i.e., 1) and `pIC50 < 9.5` as ineffective (i.e., 0). Append this as a new column called `is_effective` to the `jak2` dataframe.

In [ ]:
########## Code ##########


########## Code ##########

Split the data into 10 folds. For each fold, train on the other 9 folds. Validate on the last fold and record your prediction.

In [ ]:
########## Code ##########


########## Code ##########

Now, classify this last fold predictions into True Positives (TP), True Negatives (TN), False Positives (FP) and False Negatives (FN).

In [ ]:
########## Code ##########


########## Code ##########

Plot the 2D t-SNE embeddings (`perplexity=30`) colored by the four classification classes from the last block.

In [ ]:
########## Code ##########


########## Code ##########

What pattern do you observe?

**Answer:**

### 1.5 (10 points) UMAP analysis on Molecular Fingerprints

Perform UMAP on the obtained principal components. Plot the results. Label effective/ineffective just as above.

In [ ]:
########## Code ##########


########## Code ##########

### 1.6 (15 points, Grad only) Visualize latent clusters for structure similarity

First run DBSCAN on only active molecules and visualize with labels. To add the labels, you can pick the coordinates of the first molecule in each cluster.

In [ ]:
########## Code ##########

fig, ax = plt.subplots(figsize=(6,6))

labels =

ax.scatter(, , c=labels)

# add labels as text
for label in np.unique(labels):
    idx = np.argwhere(labels == label)[0]
    x =
    y =
    ax.text(x, y, label, size=20)

########## Code ##########

Now pick one of the clusters (list of SMILES strings) by setting the label and visualize.

In [ ]:
########## Code ##########

cluster =

# visualize all molecules in cluster
mol_list = []
for mol in cluster:
    mol_list.append(Chem.MolFromSmiles(mol))
Draw.MolsToGridImage(mol_list)

########## Code ##########

Comment on the similarity of structures within the cluster. Explore by trying a few different clusters.

**Answer:**